## **Week 6 Monday: API Error Resilience: Retries, Exponential Backoff, Rate Limits**

**Objective:** Learn to build robust applications that can gracefully handle API failures, transient errors, 
and rate limits using intelligent retry mechanisms.

**Tools:** `requests`, `urllib3`

### **1. Introduction: Why APIs Fail**

In a perfect world, every API call would succeed instantly. In reality, failures are inevitable in distributed systems. 

They can stem from:

- **Network Issues:** Temporary glitches, DNS failures, packet loss.

- **Server Overload:** The API provider is under heavy load and cannot process your request immediately.

- **Rate Limiting:** You have exceeded the number of requests allowed in a specific time window (e.g., 100 requests per minute).

- **Transient Errors:** Short-lived failures that often resolve themselves (e.g., a database connection timing out momentarily).

If your application does not handle these gracefully, it can lead to a poor user experience, data loss, or even contribute to cascading failures.

### 1.1 How They Manifest as HTTP Status Codes

- **500 Internal Server Error:** A generic catch-all, but often transient.

- **502 Bad Gateway:** One server got an invalid response from another server it was accessing (e.g., a load balancer and a backend).

- **503 Service Unavailable:** The server is explicitly saying it's down for maintenance or is overloaded.

- **504 Gateway Timeout:** A server further up the chain took too long to respond.

- **429 Too Many Requests:** You are being rate-limited.

### **2) Limitations of using only `requests` without resilience**

A bare call like `requests.get(url ...)` has several limitations:

- Makes **one attempt**; if it hits a transient error, your program fails unless you add handling.

- Lacks a consistent **retry policy**, **backoff timing**, or **rate‑limit awareness**.

- Can contribute to **thundering herd** on a busy API if you hammer it with immediate retries.

**The Naive Retry (What NOT to do):**

A simple loop without a delay can overwhelm a struggling server, turning a partial failure into a full outage.

```python
import requests

# This is a BAD practice
for i in range(5):
    try:
        response = requests.get("https://some-api.com/data")
        response.raise_for_status()
        break # Success, exit the loop
    except requests.exceptions.RequestException:
        print("Request failed, retrying...")
        # No wait time! This bombards the server.
        continue



import requests

# This is a BAD practice
ids = [1,2,3,4,5,..,1000]
for id in ids:
    try:
        params = {"id": id}
        response = requests.get("https://some-api.com/data", params=params)
        response.raise_for_status()
        break # Success, exit the loop
    except requests.exceptions.RequestException:
        print("Request failed, retrying...")
        # No wait time! This bombards the server.
        continue

```

### **3. Core Resilience Concepts**

### 3.1. Timeouts: The First Line of Defense

- timeout parameter in the Python `requests` library is used to limit the maximum amount of time, in seconds, that a request will wait for the server to respond before aborting the connection and raising a `requests.exceptions.Timeout` exception

- Always set timeouts. A request without a timeout can hang forever, consuming resources in your application.

In [1]:
# This will raise a `requests.exceptions.Timeout` after 5 seconds.
import requests
try:
    response = requests.get("https://httpbin.org/delay/10", timeout=5, )
except requests.exceptions.Timeout:
    print("The request timed out. It's better than hanging forever!")

The request timed out. It's better than hanging forever!


### 3.2. Retries: Trying Again

The basic idea is to retry a failed request. However, **not all failures should be retried**.

**Do Retry:** `500 Internal Server Error`, `502 Bad Gateway`, `503 Service Unavailable`, `504 Gateway Timeout`, `429 Too Many Requests` (after waiting).

**Do NOT Retry:** `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found`. These are client errors that won't succeed with an identical retry.

### 3.3. Exponential Backoff: The Smart Wait

It controls how long to wait between retry attempts.

Instead of retrying immediately or with a fixed delay, exponential backoff increases the wait time before each subsequent retry. This gives the API server time to recover from overload.

$\text{sleep time} = \text{backoff\_factor} \times 2^{\,n-1}$


For example, With `backoff_factor=1`, if `Retry(total=4, backoff_factor=1)`:

- **1st retry:** Wait 1 second (1 × 2⁰) = 1 sec

- **2nd retry:** Wait 2 seconds (1 × 2¹) = 2 Sec

- **3rd retry:** Wait 4 seconds (1 × 2²) = 4 Sec

- **4th retry:** Wait 8 seconds (1 × 2³) = 8 sec

### 3.4. Handling Rate Limits Specifically

APIs often communicate limits via response headers. A sophisticated client would read these.

In [2]:
import time
import requests

url = "https://httpbin.org/status/429"  # This endpoint simulates rate limiting
retries = 3  # try up to 3 times

for attempt in range(retries):
    resp = requests.get(url, timeout=5)
    if resp.status_code == 429:  # rate limited
        wait = int(resp.headers.get("Retry-After", "2"))  # default: 2s
        print(f"Rate limited (429). Waiting {wait}s… (try {attempt+1}/{retries})")
        time.sleep(wait)
        continue
    break  # got a non-429 response, exit loop

# 429 = Too Many Requests from a user (rate limiting), 
# servers may include Retry-After to say how long to wait before trying again.

Rate limited (429). Waiting 2s… (try 1/3)
Rate limited (429). Waiting 2s… (try 2/3)
Rate limited (429). Waiting 2s… (try 3/3)


### **4) Practical retries with `requests`** 

You can attach an **`HTTPAdapter`** with a **`Retry`** strategy to a `Session`.

Key configuration options:

- `total` — how many total tries (initial + retries)

- `status_forcelist` — which HTTP codes to retry (e.g., 429, 500, 502, 503, 504)

- `allowed_methods` — only retry idempotent methods by default (e.g., GET, HEAD)

- `backoff_factor` — enables exponential backoff between attempts

HTTPAdapter is a tool that tells Python how to handle the technical details (i.e. retries, backoff_factor, .etc) of sending HTTP requests to a server. 

In [3]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def create_simple_session():
    """Create a session with retry logic - no globals, no pool settings"""
    session = requests.Session()
    
    # Setup retry strategy
    retry_strategy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "HEAD"], # tells the retry system to retry only these HTTP methods. No POST why? May create duplicates
    raise_on_status=False,  
)
    
    # Attach retry logic to session
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter) # tells the session to use this adapter for https:// URLs
    session.mount("http://", adapter) # tells the session to use this adapter for http:// URLs


    return session

## How to use session with retries

In [4]:
# Test with a service that returns 503 errors
session = create_simple_session()
resp = session.get("https://httpbin.org/status/503", timeout=5)  # will retry with backoff
print("Status after retries:", resp.status_code)

Status after retries: 503


### 4.1 Why do we need a Session?

A `Session` object in `requests` allows you to persist certain parameters across requests. It also manages and persists cookies, connection pooling, and settings like retries. Think of it as a "browser tab" for your code.

Without a session, each request is independent, and you would have to keep repeatting headers or set up retries for each request individually.



### 4.2 What is HTTPAdapter?

HTTPAdapter is like a "plugin" that adds extra functionality to your session. It sits between your code and the actual HTTP connection:

```text
Your Code  -->  Session  -->  HTTPAdapter  -->  HTTP Connection (Internet)
```

In our implementation, the HTTPAdapter is configured to handle retries with exponential backoff.


### **5. Common Mistakes & Best Practices**

| Mistake | Consequence | Best Practice |
|---------|-------------|---------------|
| No timeouts | Application hangs indefinitely | Always set a reasonable timeout |
| Retrying every error | Wasting resources on client errors (4xx) | Only retry specific retryable errors (5xx, 429, timeouts) |
| Immediate, rapid retries | Overwhelming the API server | Use exponential backoff with jitter |
| Infinite retries | Application gets stuck in a loop | Set a maximum number of retries |
| Ignoring 429/Retry-After | Getting banned by API provider | Respect rate limits and use Retry-After header |

### **6. Quiz**
### 내가 적은 Answer

1. What HTTP status code indicates rate limiting?

Answer: 20 

2. With `backoff_factor=2`, how long will the 3rd retry wait?

Answer: 8: (2 * 2^(3-1)) = 8

3. Which status codes should you generally NOT retry?

Answer: 429 

4. Why is exponential backoff better than fixed delays?

Answer: 
'Exponential backoff' controls how long to wait between retry attempts.
Exponential backoff increases the wait time before each subsequent retry, instead of retrying immediately or with a fixed delay. 
It gives the API server time to recover from overload.

5. What's the purpose of the `Retry-After` header?

Answer: 'Retry-After' HTTP header shows how long a user agent should wait before making a follow-up request to a server

### 7. Class Exercise

**Task:** Create a resilient API client that can handle different types of failures.

1. Test your retry session with these endpoints:

   - `https://httpbin.org/status/500`

   - `https://httpbin.org/status/429`

   - `https://httpbin.org/status/200` (should work on first try)

2. Modify the retry strategy to:

   - Retry only on 500 and 503 status codes

   - Use a backoff_factor of 0.5
   
   - Allow only 2 total retries

3. **Bonus:** Add a custom header to track retry attempts

In [5]:
# Your exercise solution here

# Step 1: Test with different endpoints

# Step 2: Create modified retry strategy

# Step 3: Bonus - Add retry tracking
